Bước 1: Import thư viện

In [1]:
# 1. Thư viện hệ thống và tính toán 
import os
import sys
import torch
import numpy as np
import pandas as pd

# 2. Thư viện NLP và Model (Hugging Face)
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup, 
    pipeline,
)
from datasets import load_dataset

# 3. Thư viện xử lý tiếng Việt và đánh giá
from underthesea import text_normalize, word_tokenize
import evaluate

# 4. Thư viện giám sát và hiển thị
import wandb
from tqdm.auto import tqdm
import gc
import matplotlib.pyplot as plt
import gradio as gr
# 5. Kích hoạt TensorBoard trong Notebook
%load_ext tensorboard

# 6. Kiểm tra CUDA có khả dụng không
cuda_available = torch.cuda.is_available()
print(f"1. CUDA Available: {cuda_available}")

if cuda_available:
    # 7. Số lượng GPU nhận diện được
    print(f"2. Số lượng GPU: {torch.cuda.device_count()}")
    
    # 8. Tên chính xác của Card đồ họa
    print(f"3. Tên GPU: {torch.cuda.get_device_name(0)}")
    
    # 9. Kiểm tra khả năng tính toán 
    print(f"4. Compute Capability: {torch.cuda.get_device_capability(0)}")
    
    # 10. Kiểm tra bộ nhớ VRAM 
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"5. Tổng bộ nhớ VRAM: {total_memory:.2f} GB")
else:
    print("Môi trường chưa nhận GPU. Vui lòng kiểm tra lại bản cài PyTorch.")
    
    import gc

# 11. Xóa các biến mô hình cũ nếu tồn tại
if 'model' in locals():
    del model
if 'trainer' in locals():
    del trainer

# 12. Ép Python giải phóng bộ nhớ
gc.collect()

# 13. Ép PyTorch dọn sạch VRAM
torch.cuda.empty_cache()

print(f"VRAM hiện tại đã được dọn dẹp. Đang trống: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")



c:\Users\MSI\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. CUDA Available: True
2. Số lượng GPU: 1
3. Tên GPU: NVIDIA GeForce RTX 5080
4. Compute Capability: (12, 0)
5. Tổng bộ nhớ VRAM: 17.09 GB
VRAM hiện tại đã được dọn dẹp. Đang trống: 17.09 GB


Bước 2: Tải Dataset

In [2]:
# 1. Khai báo tên mô hình và tải Tokenizer
model_name = "vinai/bartpho-syllable"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Tải bộ dữ liệu VietNews (Lấy thử 5000 bài để chạy test trước cho nhanh)
print("Đang tải dữ liệu...")
raw_datasets = load_dataset("Yuhthe/vietnews", "default")

# Xem thử cấu trúc dữ liệu
print(raw_datasets)
print("\nVí dụ 1 bài báo:")
print(f"Gốc: {raw_datasets['train'][0]['article'][:200]}...")
print(f"Tóm tắt: {raw_datasets['train'][0]['abstract']}")

Đang tải dữ liệu...
DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'abstract', 'article'],
        num_rows: 99134
    })
    validation: Dataset({
        features: ['guid', 'title', 'abstract', 'article'],
        num_rows: 22184
    })
    test: Dataset({
        features: ['guid', 'title', 'abstract', 'article'],
        num_rows: 22498
    })
})

Ví dụ 1 bài báo:
Gốc: Ngày 27/3 , Cơ quan Cảnh sát điều tra Công an TP. Hưng Yên , tỉnh Hưng Yên cho biết , đơn vị vừa ra quyết định khởi tố vụ án , khởi tố bị can đối với đối tượng Mai Văn Thương ( SN 1989 , trú tại đội 1...
Tóm tắt: Với bản tính ham chơi , lười làm , có nhiều tiền án tiền sự , lại nghiện ma tuý , Thương đã đột nhập vào nhà chú ruột để trộm hơn 1 tạ thóc và hơn 8 triệu đồng mang đi tiêu xài .


Bước 3: Hàm tiền xử lý

In [3]:
def preprocess_function(examples):
# 1. Chuẩn hóa tiếng Việt và đưa về dạng âm tiết cho BARTpho
    inputs = [text_normalize(doc) for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")

# 2. Xử lý nhãn (đoạn tóm tắt)
    targets = [text_normalize(target) for target in examples["abstract"]]
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3. Chạy tiền xử lý trên toàn bộ dataset
print("Đang tiến hành Tokenize dữ liệu...")
tokenized_datasets = raw_datasets.map(
    preprocess_function, 
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

print("Tiền xử lý hoàn tất!")
print(f"Dữ liệu mẫu sau khi Tokenize: {tokenized_datasets['train'][0]['input_ids'][:10]}...")

Đang tiến hành Tokenize dữ liệu...
Tiền xử lý hoàn tất!
Dữ liệu mẫu sau khi Tokenize: [0, 698, 1654, 1779, 23, 4, 1221, 65, 1400, 347]...


Bước 4: Khởi tạo Mô hình và Data Collator & Thiết lập tham số huấn luyện & Kích hoạt quá trình Huấn luyện

4.1: Khởi tạo ROUGE trong TRAINER

In [ ]:
# Nạp metric ROUGE
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Giải mã (decode) các token thành văn bản
    decoded_preds = tokenizer.batch_decode(preds, skip_special_objects=True)
    # Thay thế -100 trong labels (các token được bỏ qua) bằng pad_token
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_objects=True)

    # Tối ưu cho tiếng Việt: ROUGE hoạt động tốt hơn khi được tokenize
    # Dùng underthesea 
    decoded_preds = [" ".join(word_tokenize(text)) for text in decoded_preds]
    decoded_labels = [" ".join(word_tokenize(text)) for text in decoded_labels]

    # Tính toán ROUGE
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # Làm tròn kết quả
    result = {key: round(value * 100, 4) for key, value in result.items()}
    return result

4.2: Chạy

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "./bartpho_vietnews_checkpoints/checkpoint-12000"

# Load model 
model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint_path,
    use_safetensors=True,
).to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bartpho_vietnews_checkpoints",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    learning_rate=2e-5,
    weight_decay=0.05,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    fp16=True,
    bf16=False,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,

    max_grad_norm=1.0,

    logging_steps=50,
    report_to="tensorboard",
    predict_with_generate=True,
    optim="adamw_torch",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"].select(range(100)), # chỉ lấy 100 mẫu để đánh giá nhanh
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

gc.collect()
torch.cuda.empty_cache()

print(f" Resume FP16 từ: {checkpoint_path}")
trainer.train(resume_from_checkpoint=checkpoint_path)


C:\Users\MSI\AppData\Local\Temp\ipykernel_20016\3243859424.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


🏁 Resume FP16 từ: ./bartpho_vietnews_checkpoints/checkpoint-12000


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
12500,0.396100,0.446723,22.886100,9.704700,16.789800,16.739900
13000,0.395700,0.443383,23.016700,10.221200,16.869000,16.840600
13500,0.395900,0.441336,22.584700,9.754200,16.751100,16.747700
14000,0.386000,0.440483,23.240200,10.171900,17.293700,17.291100
14500,0.399900,0.440392,22.461200,9.500900,16.699200,16.674000
15000,0.400000,0.440446,23.011500,9.990900,16.868500,16.827100
15500,0.396300,0.438197,23.155900,10.250000,17.150900,17.152900
16000,0.399500,0.437253,22.697500,9.620900,16.700600,16.698000
16500,0.389000,0.437329,22.941000,9.970000,16.992600,16.955500
17000,0.397700,0.436717,22.720400,9.707600,16.915000,16.906600


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=18588, training_loss=0.14002851498929922, metrics={'train_runtime': 5766.2045, 'train_samples_per_second': 51.577, 'train_steps_per_second': 3.224, 'total_flos': 6.445087272437023e+17, 'train_loss': 0.14002851498929922, 'epoch': 3.0})

Bước 5: Lưu mô hình

In [ ]:
# 1. Đường dẫn đến checkpoint 18000 của bạn
checkpoint_path = "./bartpho_vietnews_checkpoints/checkpoint-18588"
# 2. Tên thư mục bạn muốn lưu mô hình cuối cùng
final_model_path = "./BARTpho_VietNews_Final"

# 3. Tạo thư mục nếu chưa có
if not os.path.exists(final_model_path):
    os.makedirs(final_model_path)

print(f"🔄 Đang tải mô hình từ {checkpoint_path}...")
# 4. Tải model và tokenizer từ checkpoint 18000
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# 5. Lưu lại chính thức
print(f"💾 Đang lưu mô hình vào {final_model_path}...")
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("✅ Đã lưu thành công! Bạn có thể dùng thư mục này để Deploy.")

🔄 Đang tải mô hình từ ./bartpho_vietnews_checkpoints/checkpoint-18588...
💾 Đang lưu mô hình vào ./BARTpho_VietNews_Final-1...
✅ Đã lưu thành công! Bạn có thể dùng thư mục này để Deploy.


Bước 6: Test mô hình

In [7]:
# 1. Khởi tạo pipeline tóm tắt từ checkpoint cuối cùng
summarizer = pipeline(
    "summarization", 
    model="./BARTpho_VietNews_Final", 
    tokenizer=tokenizer,
    device=0 # Chạy trên RTX 5080 cho nhanh
)

# Đoạn văn mẫu về VinFast và thị trường xe điện (Phức tạp hơn)
complex_text = """
VinFast vừa công bố doanh thu quý IV năm 2025 với mức tăng trưởng vượt bậc tại thị trường Mỹ và Indonesia. 
Theo báo cáo, hãng xe điện Việt Nam đã bàn giao hơn 50.000 xe trên toàn cầu chỉ trong vòng 3 tháng, 
đánh dấu cột mốc quan trọng trong chiến lược bành trướng ra quốc tế. Tuy nhiên, thách thức lớn nhất 
vẫn là sự cạnh tranh khốc liệt từ các đối thủ Trung Quốc và chính sách thuế nhập khẩu thay đổi tại một số thị trường.
Để đối phó, VinFast dự kiến sẽ đẩy mạnh xây dựng nhà máy tại chỗ nhằm tối ưu chi phí vận chuyển và nhận ưu đãi thuế.
"""

# Chạy tóm tắt
result = summarizer(
    complex_text, 
    max_new_tokens=100, 
    min_length=40,       # Ép AI phải viết dài hơn một chút để không bỏ sót ý
    do_sample=False,     # Tắt sampling để AI tập trung vào các từ khóa có xác suất cao nhất
    num_beams=5,         # Dùng Beam Search để tìm tổ hợp câu hay nhất
    no_repeat_ngram_size=3 # Ngăn chặn việc lặp lại các cụm từ
)

print("Kết quả tóm tắt cho bài báo kinh tế:")
print(result[0]['summary_text'])

Device set to use cuda:0
Your min_length=40 must be inferior than your max_length=20.


Kết quả tóm tắt cho bài báo kinh tế:
VinFast dự kiến sẽ đẩy mạnh xây dựng nhà máy tại chỗ nhằm tối ưu chi phí vận chuyển và nhận ưu đãi thuế từ chính sách thuế nhập khẩu thay đổi tại một số thị trường .


Bước 7: Giao diện

In [8]:
def summarize_vietnamese(text):
    # Sử dụng pipeline đã khởi tạo ở trên
    result = summarizer(
        text, 
        max_new_tokens=100, 
        min_length=30, 
        do_sample=False, 
        num_beams=5
    )
    return result[0]['summary_text'].replace(" .", ".")

# Giao diện người dùng
demo = gr.Interface(
    fn=summarize_vietnamese,
    inputs=gr.Textbox(lines=10, placeholder="Dán bài báo tiếng Việt vào đây..."),
    outputs=gr.Textbox(lines=10, label="Bản tóm tắt của AI"),
    title="AI Tóm Tắt Tin Tức Tiếng Việt (BARTpho)",
    description="Ứng dụng sử dụng mô hình bạn vừa huấn luyện tại Step 18000/18500."
)

demo.launch(share=True) # share=True sẽ tạo một link web công khai trong 72h

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://845eb20c736257db78.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
Your min_length=30 must be inferior than your max_length=20.
